In [1]:
# Preparation. Set USE_LOCAL_EXPORT=True to read a folder from
# scripts/export_labeling_run.py; False uses live Databricks MLflow.
# SOURCE_RUN_ID is only needed the first time an episode is opened
# (when gold has no "pending" yet).
import json
from pathlib import Path

from drzavni_posao import MediaStore, MlflowTracker, Stage
from drzavni_posao.transcript import Transcript

USE_LOCAL_EXPORT = True
SOURCE_RUN_ID = "1c519fba673841fa9188addf9a720ae6"
EPISODE_KEY = "1"
REDO = []  # Zero-based indexes in gold["segments"].
CHARACTERS = ["Djordje Cvarkov", "Dragan Torbica", "Boskic"]

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CATALOG_PATH = REPO_ROOT / "data" / "youtube_episodes.json"
if USE_LOCAL_EXPORT:
    AUDIO_PATHS = REPO_ROOT / "downloads"
    LABEL_DIR = REPO_ROOT / "data" / "labels"
    RUN_EXPORT = REPO_ROOT / "data" / "mlflow_exports" / SOURCE_RUN_ID
    EXPERIMENT_NAME = None
else:
    AUDIO_PATHS = Path("/Volumes/luka_erceg/drzavni_posao/audios")
    LABEL_DIR = Path(
        "/Workspace/Users/luka.erceg@databricks.com/drzavni_posao/labels"
    )
    RUN_EXPORT = None
    EXPERIMENT_NAME = (
        "/Users/luka.erceg@databricks.com/drzavni_posao/transcription"
    )

tracker = None if USE_LOCAL_EXPORT else MlflowTracker(EXPERIMENT_NAME)
label_path = LABEL_DIR / f"{EPISODE_KEY}.json"
gold = (
    json.loads(label_path.read_text(encoding="utf-8"))
    if label_path.exists()
    else None
)
needs_source = gold is None or "pending" not in gold


def stage_transcript(stage: Stage) -> Transcript:
    if USE_LOCAL_EXPORT:
        path = (
            RUN_EXPORT
            / "episodes"
            / EPISODE_KEY
            / stage.value
            / "transcript.json"
        )
        if not path.is_file():
            raise FileNotFoundError(
                f"Missing {path}. Export the run or pick an episode that has "
                f"{stage.value}."
            )
        return Transcript.from_dict(
            json.loads(path.read_text(encoding="utf-8"))
        )
    source = tracker.episode_run_states(SOURCE_RUN_ID)[EPISODE_KEY]
    return tracker.stage_transcript(source.run_id, stage)


def load_proposal():
    if USE_LOCAL_EXPORT:
        config_path = RUN_EXPORT / "config.json"
        if not config_path.is_file():
            raise FileNotFoundError(
                f"Missing export at {RUN_EXPORT}. Run "
                f"scripts/export_labeling_run.py --run-id {SOURCE_RUN_ID}"
            )
        config = json.loads(config_path.read_text(encoding="utf-8"))
    else:
        config = tracker._configuration(SOURCE_RUN_ID)
    return {key: config[key] for key in ("audio", "transcription")}


if needs_source:
    proposal = load_proposal()
    if gold is not None and gold["proposal"] != proposal:
        raise ValueError(
            "This episode was labeled from a different ASR configuration"
        )
else:
    proposal = gold["proposal"]

media = MediaStore(
    AUDIO_PATHS,
    LABEL_DIR,
    catalog_path=CATALOG_PATH,
    **proposal["audio"],
)
episode = media.episode(EPISODE_KEY)
trim_start = proposal["audio"]["trim_start"]


def overlap(segment, segment_trim, label):
    return max(
        0,
        min(segment["end"] + segment_trim, label["end"])
        - max(segment["start"] + segment_trim, label["start"]),
    )


episode.transcript = stage_transcript(Stage.TRANSCRIPTION)
if gold is None:
    gold = {
        "schema_version": 1,
        "episode_key": EPISODE_KEY,
        "timebase": "raw_audio",
        "proposal": proposal,
        "segments": [],
    }
gold["pending"] = [
    segment
    for segment in episode.transcript.segments
    if not any(overlap(segment, trim_start, label) for label in gold["segments"])
]

for index in dict.fromkeys(REDO):
    if not isinstance(index, int) or not 0 <= index < len(gold["segments"]):
        raise IndexError(f"Invalid REDO index {index!r}")

for speaker, candidates in gold.pop("gallery_candidates", {}).items():
    for segment in gold["segments"]:
        if segment["speaker"] == speaker and any(
            segment["start"] == candidate["start"]
            and segment["end"] == candidate["end"]
            for candidate in candidates
        ):
            segment["gallery_candidate"] = True


def save():
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text(
        json.dumps(gold, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


def gold_speaker(segment, candidate_trim_start=0):
    matches = [
        (overlap(segment, candidate_trim_start, label), label["speaker"])
        for label in gold["segments"]
    ]
    seconds, speaker = max(
        matches, key=lambda match: match[0], default=(0, None)
    )
    return speaker if seconds else None


save()
audio = episode.audio()
print(
    f"{len(gold['segments'])} labeled, {len(gold['pending'])} pending, "
    f"{sum(segment.get('gallery_candidate', False) for segment in gold['segments'])} "
    "gallery candidates"
)
print(f"mode={'local_export' if USE_LOCAL_EXPORT else 'mlflow'}")
print(f"audio={episode.audio_path}")
print(f"labels={label_path}")
if needs_source:
    print(f"source={RUN_EXPORT if USE_LOCAL_EXPORT else SOURCE_RUN_ID}")
else:
    print("source=existing labels")


47 labeled, 70 pending, 0 gallery candidates
mode=local_export
audio=/Users/luka.erceg/drzavni_posao/downloads/season-1/DRŽAVNI POSAO [HQ] - Ep.1： Rakija (24.09.2012.).mp3
labels=/Users/luka.erceg/drzavni_posao/data/labels/1.json
source=existing labels


In [ ]:
# Interactive character-labeling reviewer.
import ipywidgets as widgets
from IPython.display import Audio, clear_output, display

from drzavni_posao.constants import ANONYMOUS_LABEL, SR

BOUNDARY_CONTEXT_SECONDS = 2.0
segments = episode.transcript.segments
audio_duration = len(audio) / SR
output = widgets.Output()
clip_output = widgets.Output()
status = widgets.HTML()
boundary_status = widgets.HTML()
progress = widgets.IntProgress(
    min=0,
    max=len(segments),
    description="Labeled",
    bar_style="success",
)
boundary_slider = widgets.FloatRangeSlider(
    value=(0.0, min(audio_duration, 1.0)),
    min=0.0,
    max=audio_duration,
    step=0.05,
    description="Start / end",
    readout_format=".2f",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)
speaker_values = [*CHARACTERS, ANONYMOUS_LABEL]
speaker_choice = widgets.ToggleButtons(
    options=[(name, name) for name in speaker_values],
    value=None,
    description="Character",
)
gallery_choice = widgets.ToggleButton(
    value=False,
    description="Gallery candidate",
    icon="check",
)
previous_button = widgets.Button(description="Previous", icon="arrow-left")
replay_button = widgets.Button(description="Replay cut", icon="refresh")
next_button = widgets.Button(description="Next", icon="arrow-right")
save_button = widgets.Button(
    description="Save & Next",
    button_style="success",
    icon="save",
)
navigation = [previous_button, replay_button, next_button, save_button]


def label_match(segment_index):
    for label_index, label in enumerate(gold["segments"]):
        if label.get("source_segment_index") == segment_index:
            return label_index, label

    segment = segments[segment_index]
    matches = [
        (overlap(segment, trim_start, label), label_index, label)
        for label_index, label in enumerate(gold["segments"])
        if "source_segment_index" not in label
    ]
    seconds, label_index, label = max(matches, default=(0, None, None))
    return (label_index, label) if seconds else (None, None)


def labeled_count():
    return sum(label_match(index)[0] is not None for index in range(len(segments)))


def pending_indexes():
    return [
        index for index in range(len(segments)) if label_match(index)[0] is None
    ]


redo_labels = set(REDO)
redo_positions = [
    index for index in range(len(segments)) if label_match(index)[0] in redo_labels
]
unreviewed = pending_indexes()
position = redo_positions[0] if redo_positions else unreviewed[0] if unreviewed else 0


def update_gallery_availability(change):
    gallery_choice.disabled = change["new"] in (None, ANONYMOUS_LABEL)
    if gallery_choice.disabled:
        gallery_choice.value = False


def update_boundary_status(_=None):
    if not segments:
        return
    start, end = boundary_slider.value
    segment = segments[position]
    boundary_status.value = (
        f"Cut <b>{start:.2f}–{end:.2f}s</b> "
        f"(start {start - segment['start']:+.2f}s, "
        f"end {end - segment['end']:+.2f}s from ASR)"
    )


def play_cut(autoplay=True):
    with clip_output:
        clear_output(wait=True)
        start, end = boundary_slider.value
        left = max(0, int(start * SR))
        right = min(len(audio), int(end * SR))
        display(Audio(audio[left:right].copy(), rate=SR, autoplay=autoplay))


def show():
    with output:
        clear_output(wait=True)
        if not segments:
            print("No transcript segments to label.")
            for control in [
                boundary_slider,
                speaker_choice,
                gallery_choice,
                *navigation,
            ]:
                control.disabled = True
            return

        segment = segments[position]
        _, saved = label_match(position)
        speaker_choice.value = (
            saved["speaker"]
            if saved and saved.get("speaker") in speaker_values
            else None
        )
        gallery_choice.value = bool(saved and saved.get("gallery_candidate"))

        cut_start = saved["start"] - trim_start if saved else segment["start"]
        cut_end = saved["end"] - trim_start if saved else segment["end"]
        cut_start = max(0.0, min(cut_start, audio_duration))
        cut_end = max(cut_start + boundary_slider.step, min(cut_end, audio_duration))
        window_start = max(
            0.0,
            min(segment["start"], cut_start) - BOUNDARY_CONTEXT_SECONDS,
        )
        window_end = min(
            audio_duration,
            max(segment["end"], cut_end) + BOUNDARY_CONTEXT_SECONDS,
        )
        boundary_slider.min = 0.0
        boundary_slider.max = audio_duration
        boundary_slider.value = (cut_start, cut_end)
        boundary_slider.min = window_start
        boundary_slider.max = window_end
        update_boundary_status()

        reviewed = labeled_count()
        progress.value = reviewed
        print(
            f"{position + 1}/{len(segments)}  labeled {reviewed}/{len(segments)}  "
            f"ASR {segment['start']:.2f}–{segment['end']:.2f}s"
        )
        print(segment.get("text", "").strip())

    previous_button.disabled = position == 0
    next_button.disabled = position == len(segments) - 1
    status.value = (
        f"<b>Saved: {saved['speaker']}</b>"
        if saved
        else "<span style='color:#777'>Not labeled</span>"
    )
    play_cut()


def step(delta):
    global position
    position = max(0, min(len(segments) - 1, position + delta))
    show()


def save_current(_):
    global position
    speaker = speaker_choice.value
    if speaker is None:
        status.value = "<b style='color:#b00'>Choose a character.</b>"
        return

    cut_start, cut_end = boundary_slider.value
    label = {
        "start": cut_start + trim_start,
        "end": cut_end + trim_start,
        "speaker": speaker,
        "source_segment_index": position,
    }
    if gallery_choice.value and speaker != ANONYMOUS_LABEL:
        label["gallery_candidate"] = True

    label_index, _ = label_match(position)
    if label_index is None:
        gold["segments"].append(label)
    else:
        gold["segments"][label_index] = label
    gold["segments"].sort(key=lambda item: (item["start"], item["end"]))
    gold["pending"] = [segments[index] for index in pending_indexes()]
    save()

    unreviewed = pending_indexes()
    if unreviewed:
        position = next(
            (index for index in unreviewed if index > position),
            unreviewed[0],
        )
    else:
        position = min(position + 1, len(segments) - 1)
    show()


speaker_choice.observe(update_gallery_availability, names="value")
boundary_slider.observe(update_boundary_status, names="value")
previous_button.on_click(lambda _: step(-1))
replay_button.on_click(lambda _: play_cut())
next_button.on_click(lambda _: step(1))
save_button.on_click(save_current)
display(
    widgets.VBox(
        [
            progress,
            output,
            boundary_slider,
            boundary_status,
            clip_output,
            speaker_choice,
            widgets.HBox([gallery_choice, status]),
            widgets.HBox(navigation),
        ]
    )
)
show()


In [4]:
# Score character assignment against this episode's gold labels.
# Local export: uses character_assigner/ from RUN_EXPORT (ASSIGNMENT_RUN_ID
# defaults to SOURCE_RUN_ID). Live MLflow: set ASSIGNMENT_RUN_ID to any finished
# assignment parent run.
import pandas as pd
from IPython.display import display

ASSIGNMENT_RUN_ID = SOURCE_RUN_ID if USE_LOCAL_EXPORT else ""
if not ASSIGNMENT_RUN_ID:
    raise ValueError("Set ASSIGNMENT_RUN_ID to a finished assignment parent run")

if USE_LOCAL_EXPORT:
    if ASSIGNMENT_RUN_ID != SOURCE_RUN_ID:
        raise ValueError(
            "Local export mode scores the exported SOURCE_RUN_ID only; "
            f"got ASSIGNMENT_RUN_ID={ASSIGNMENT_RUN_ID!r}"
        )
    assignment_config = json.loads(
        (RUN_EXPORT / "config.json").read_text(encoding="utf-8")
    )
    episode.transcript = stage_transcript(Stage.CHARACTER_ASSIGNER)
else:
    if tracker is None:
        tracker = MlflowTracker(EXPERIMENT_NAME)
    assignment_config = tracker._configuration(ASSIGNMENT_RUN_ID)
    assignment = tracker.episode_run_states(ASSIGNMENT_RUN_ID)[EPISODE_KEY]
    tracker.restore_stage(episode, assignment.run_id, Stage.CHARACTER_ASSIGNER)

assignment_trim_start = assignment_config["audio"]["trim_start"]

rows = []
for segment in episode.transcript.segments:
    expected = gold_speaker(segment, assignment_trim_start)
    if expected is not None:
        rows.append(
            {
                "start": segment["start"],
                "end": segment["end"],
                "gold": expected,
                "assigned": segment.get("speaker") or "[unassigned]",
                "text": segment.get("text", "").strip(),
            }
        )

results = pd.DataFrame(rows)
if results.empty:
    print("No assignment segments overlap the gold labels.")
else:
    print(f"accuracy: {(results['gold'] == results['assigned']).mean():.1%}")
    display(pd.crosstab(results["gold"], results["assigned"]))
    display(results[results["gold"] != results["assigned"]])


accuracy: 95.4%


assigned,Boskic,Djordje Cvarkov,Dragan Torbica
gold,,,
Boskic,25,1,1
Djordje Cvarkov,0,40,3
Dragan Torbica,0,0,39


,start,end,gold,assigned,text
16,49.198,50.599,Boskic,Dragan Torbica,Ko plače?
24,66.951,68.223,Djordje Cvarkov,Dragan Torbica,Šta se uradili?
38,101.996,102.957,Djordje Cvarkov,Dragan Torbica,Šta sam zboravio?
57,154.904,155.975,Boskic,Djordje Cvarkov,Što su moji brata ote
105,293.091,296.474,Djordje Cvarkov,Dragan Torbica,A to je tamo.


In [ ]:
# Build the expanded speaker gallery from every labeled episode.
gallery_path = Path("data/speaker_galleries/expanded.json")
gallery = {}

for path in sorted(Path(LABEL_DIR).glob("[0-9]*.json")):
    labels = json.loads(path.read_text(encoding="utf-8"))
    for segment in labels.get("segments", []):
        if segment.get("gallery_candidate"):
            gallery.setdefault(segment["speaker"], []).append(
                {
                    "episode_key": labels["episode_key"],
                    "start": segment["start"],
                    "end": segment["end"],
                }
            )

for clips in gallery.values():
    clips.sort(key=lambda clip: (clip["episode_key"], clip["start"]))

gallery_path.parent.mkdir(parents=True, exist_ok=True)
gallery_path.write_text(
    json.dumps(gallery, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
print(
    f"Saved {sum(map(len, gallery.values()))} clips for {len(gallery)} characters "
    f"to {gallery_path}"
)
gallery
